In [ ]:
import os
if not "src" in os.listdir(): # I use this to change my directory to access the src and data folders from in here
    os.chdir("..") 
# print(os.listdir()) # this should say something like ['README.md', 'wk5_toolbox', '.git', 'wk4_toolbox']

import networkx as nx
import markov_clustering as mc

import numpy as np
import scipy as sp
import pandas as pd
from collections import Counter

import mygene # allow us to convert from systematic to standard

import matplotlib.pyplot as plt

from src.reading_in import read_in_proteins
from src.community_finding import partition_graph, partition_graph_size, eliminate_small_communities, get_adjacent_communities



# A protein is considered a core protein if it is assigned
# to its dominant community at least this fraction of the time.
CORE_THRESHOLD = 0.99

DRIFTER_THRESHOLD = 0.05 # Min freq for a community to be considered relevant when identifying two-community drifters.
TOP_N = 10 #Number of high-degree proteins used to identify a community
MATCH_THRESHOLD = 0.4 #min similarity between comm signatures to consider them same community

from wk7_toolbox.wk7_subroutines import louvain_frequencies_using_all_members, louvain_frequencies_using_high_degree_proteins, summarise_louvain_results_method2, summarise_louvain_results_method1

FILEPATH = "data/raw/protein.links.v12.0.txt.gz"
G = read_in_proteins(include_essential_proteins=False, connected=True)

In [ ]:
G = read_in_proteins(include_essential_proteins=False, connected=True) 
largest_cc_nodes = max(nx.connected_components(G), key=len)
G_connected = G.subgraph(largest_cc_nodes).copy()

node_list = list(G_connected.nodes())  # index i corresponds to protein node_list[i]

#Decresing the inflation will make the clusters larger, This is an abritary choice
matrix = nx.to_numpy_array(G_connected)
result = mc.run_mcl(matrix, inflation=1.2)
clusters = mc.get_clusters(result)

#This part basically creates a dictionary where you input the node and output the cluster it puts it in
node_to_cluster = {}
for cluster_id, cluster in enumerate(clusters):
    for idx in cluster:
        node_to_cluster[node_list[idx]] = cluster_id

#This is the function which finds the smallest distance from a target protien to the clusters
def cluster_min_distances(G, target, node_to_cluster):
    distances_from_target = nx.shortest_path_length(G, source=target)
    cluster_min_dist = {}
    for node, dist in distances_from_target.items():
        cid = node_to_cluster[node]
        if cid not in cluster_min_dist or dist < cluster_min_dist[cid]:
            cluster_min_dist[cid] = dist
    return cluster_min_dist

# Here you can choose a protein and then choose the distance you are looking for
target_protein = "YGR220C"
cluster_min_dist = cluster_min_distances(G_connected, target_protein, node_to_cluster)
candidates = {cid: d for cid, d in cluster_min_dist.items() if 3 <= d <= 5}

print(candidates)


runs the community finding algorithm several times (use 500 as a guide), and identifies those proteins which are always assigned to the same community and those which are assigned to different
communities in diDerent realizations. The former define the community core and the latter are the drifter proteins. Drifters which are split between just two communities with roughly equal frequencies are good candidates of proteins that link the two communities.

In [ ]:
# Method 1 outline in toolbox
res = louvain_frequencies_using_all_members(G, 50, CORE_THRESHOLD, DRIFTER_THRESHOLD)
summarise_louvain_results_method1(res)

 label the communities and compare the same labels for different
realizations. do this for larger communities by identifying a community
by its high degree nodes

In [ ]:
#Method 2 challenge part of toolbox
res = louvain_frequencies_using_high_degree_proteins(G, 20, TOP_N, MATCH_THRESHOLD)
summarise_louvain_results_method2(res)
